# BGGTDM v2 — Model Training Notebook

Runs all steps in order:
1. Feature prep (EB shrinkage, target_share)
2. LR baseline + XGBoost training
3. Probability calibration (temperature + beta)
4. Final evaluation + sportsbook comparison

In [ ]:
import os
os.chdir('/Users/aaronangeles/Documents/Projects/Active/biggamegabefallacy/ml')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## Step 1 — Feature Preparation

- Computes `target_share` (missing from enriched CSV)
- Fits Beta-Binomial shrinkage on training set → `td_rate_eb`, `td_rate_eb_std`
- Returns clean X_train / X_holdout matrices

In [ ]:
from feature_prep import load_and_prepare, FEATURES

X_train, y_train, X_holdout, y_holdout, train_df, holdout_df, eb_params = load_and_prepare()

print(f'\nEB params: alpha={eb_params["alpha"]:.4f}, beta={eb_params["beta"]:.4f}')
print(f'Features ({len(FEATURES)}):', FEATURES)

In [ ]:
# Inspect EB-shrunk td_rate distribution vs raw
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train_df['td_rate_per_target'].dropna(), bins=50, edgecolor='k', alpha=0.7)
axes[0].set_title('Raw td_rate_per_target (train)')
axes[0].set_xlabel('Rate')

axes[1].hist(train_df['td_rate_eb'], bins=50, edgecolor='k', alpha=0.7, color='steelblue')
axes[1].set_title('EB-shrunk td_rate_eb (train)')
axes[1].set_xlabel('Rate')

plt.tight_layout()
plt.show()

print('td_rate_eb percentiles:')
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f'  {p}th: {np.percentile(train_df["td_rate_eb"], p):.4f}')

## Step 2 — Logistic Regression Baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_train)
X_ho_scaled = scaler.transform(X_holdout)

lr = LogisticRegression(class_weight='balanced', C=0.1, max_iter=1000, random_state=42)
lr.fit(X_tr_scaled, y_train)

lr_probs = lr.predict_proba(X_ho_scaled)[:, 1]

print('LR Baseline — 2025 Holdout')
print(f'  AUC:   {roc_auc_score(y_holdout, lr_probs):.4f}')
print(f'  Brier: {brier_score_loss(y_holdout, lr_probs):.4f}')
print(f'  LogL:  {log_loss(y_holdout, lr_probs):.4f}')
pcts = np.percentile(lr_probs, [10, 25, 50, 75, 90, 95])
print(f'  Percentiles (10/25/50/75/90/95): {pcts.round(3)}')

# LR coefficients
coefs = sorted(zip(FEATURES, lr.coef_[0]), key=lambda x: -abs(x[1]))
print('\nTop 10 LR coefficients:')
for f, c in coefs[:10]:
    print(f'  {f:<25} {c:+.4f}')

## Step 3 — XGBoost with Monotone Constraints

In [ ]:
import xgboost as xgb
from train import MONOTONE

constraints = tuple(MONOTONE[f] for f in FEATURES)

xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=20,
    scale_pos_weight=1,        # no upsampling — calibration handles class balance
    monotone_constraints=constraints,
    eval_metric='logloss',
    early_stopping_rounds=30,
    random_state=42,
)

# Use last 5 weeks of 2024 for early stopping (not 2025 holdout, not full 2024 training)
eval_mask = (train_df['season'] == 2024) & (train_df['week'] >= 14)
X_eval = X_train[eval_mask]
y_eval = y_train[eval_mask]
X_tr = X_train[~eval_mask]
y_tr = y_train[~eval_mask]

print(f'Training on {len(X_tr)} rows (2022–2024 wk1-13), eval on {len(X_eval)} rows (2024 wk14-18)')
xgb_model.fit(X_tr, y_tr, eval_set=[(X_eval, y_eval)], verbose=50)
print(f'Best iteration: {xgb_model.best_iteration}')

In [ ]:
xgb_probs_train = xgb_model.predict_proba(X_train)[:, 1]
xgb_probs_holdout = xgb_model.predict_proba(X_holdout)[:, 1]

print('XGBoost — Train')
print(f'  AUC:   {roc_auc_score(y_train, xgb_probs_train):.4f}')
print(f'  Brier: {brier_score_loss(y_train, xgb_probs_train):.4f}')

print('\nXGBoost — Holdout (2025)')
print(f'  AUC:   {roc_auc_score(y_holdout, xgb_probs_holdout):.4f}  (target > 0.730)')
print(f'  Brier: {brier_score_loss(y_holdout, xgb_probs_holdout):.4f}  (target < 0.115)')
print(f'  LogL:  {log_loss(y_holdout, xgb_probs_holdout):.4f}')
pcts = np.percentile(xgb_probs_holdout, [10, 25, 50, 75, 90, 95])
print(f'  Percentiles (10/25/50/75/90/95): {pcts.round(3)}')

# Feature importance
importances = xgb_model.feature_importances_
fi = sorted(zip(FEATURES, importances), key=lambda x: -x[1])
print('\nFeature importances:')
for feat, imp in fi:
    bar = '█' * int(imp * 200)
    print(f'  {feat:<25} {imp:.4f} {bar}')

## Step 4 — Probability Calibration

In [ ]:
from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from betacal import BetaCalibration
from utils import reliability_diagram

# Calibrate on 2024 (XGBoost eval set — not used in gradient descent)
raw_probs_val = xgb_model.predict_proba(X_eval)[:, 1]

# Temperature scaling
raw_logits_val = logit(np.clip(raw_probs_val, 1e-6, 1-1e-6))

def neg_logloss(tau):
    p = expit(raw_logits_val / tau)
    return log_loss(y_eval, p)

result = minimize_scalar(neg_logloss, bounds=(0.1, 5.0), method='bounded')
tau = result.x
print(f'Temperature scaling tau = {tau:.4f}')

ts_probs = expit(logit(np.clip(xgb_probs_holdout, 1e-6, 1-1e-6)) / tau)

# Beta calibration
bc = BetaCalibration(parameters='abm')
bc.fit(raw_probs_val.reshape(-1, 1), y_eval)
bc_probs = bc.predict(xgb_probs_holdout.reshape(-1, 1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

bm0, bf0, _ = reliability_diagram(y_holdout, xgb_probs_holdout, title='Raw XGBoost', ax=axes[0])
bm1, bf1, _ = reliability_diagram(y_holdout, ts_probs, title=f'Temp Scaling (τ={tau:.3f})', ax=axes[1])
bm2, bf2, _ = reliability_diagram(y_holdout, bc_probs, title='Beta Calibration', ax=axes[2])

plt.tight_layout()
os.makedirs('model', exist_ok=True)
plt.savefig('model/reliability_diagrams.png', dpi=120, bbox_inches='tight')
plt.show()

ts_dev = np.max(np.abs(bm1 - bf1))
bc_dev = np.max(np.abs(bm2 - bf2))
print(f'Max deviation — Temp Scaling: {ts_dev:.4f} | Beta Cal: {bc_dev:.4f}')

best_cal = 'temperature' if ts_dev <= bc_dev else 'beta'
cal_probs = ts_probs if best_cal == 'temperature' else bc_probs
print(f'Using: {best_cal} calibration')

print(f'\nFinal calibrated prob percentiles (10/25/50/75/90/95):')
pcts = np.percentile(cal_probs, [10, 25, 50, 75, 90, 95])
print(f'  {pcts.round(3)}')

## Step 5 — Final Evaluation

In [ ]:
from evaluate import td_recall_at_top_k
from utils import american_to_implied_prob, prob_to_american

auc = roc_auc_score(y_holdout, cal_probs)
brier = brier_score_loss(y_holdout, cal_probs)
ll = log_loss(y_holdout, cal_probs)
recall_top20 = td_recall_at_top_k(y_holdout, cal_probs, 0.20)

print('=== FINAL HOLDOUT METRICS (2025) ===')
checks = [
    ('ROC-AUC', auc,          0.730, 'PASS' if auc > 0.730 else 'FAIL',         '> 0.730'),
    ('Brier',   brier,        0.115, 'PASS' if brier < 0.115 else 'FAIL',        '< 0.115'),
    ('95th pct', np.percentile(cal_probs,95), 0.50,
     'PASS' if np.percentile(cal_probs,95) > 0.50 else 'FAIL', '> 0.50'),
    ('TD Recall @20%', recall_top20, 0.35, 'PASS' if recall_top20 > 0.35 else 'FAIL', '> 0.35'),
]
for name, val, tgt, status, tgt_str in checks:
    print(f'  [{status}] {name:<20} = {val:.4f}  (target {tgt_str})')
print(f'\n  Log-Loss: {ll:.4f}')

In [ ]:
# Show top predictions for 2025 holdout
results_df = holdout_df[['name', 'pos', 'team', 'season', 'week', 'scored_td']].copy()
results_df['model_prob'] = cal_probs
results_df['model_odds'] = [prob_to_american(p) for p in cal_probs]
results_df = results_df.sort_values('model_prob', ascending=False)

print('Top 25 predictions (2025 holdout):')
print(results_df.head(25)[['name', 'pos', 'team', 'week', 'model_prob', 'model_odds', 'scored_td']].to_string(index=False))

## Step 6 — Save Final Bundle

In [ ]:
os.makedirs('model', exist_ok=True)

bundle = {
    'model': xgb_model,
    'scaler': scaler,
    'features': FEATURES,
    'tau': tau,
    'beta_calibrator': bc,
    'best_calibration': best_cal,
    'alpha_eb': eb_params['alpha'],
    'beta_eb': eb_params['beta'],
    'trained_on': '2022-2024',
    'holdout_auc': auc,
    'holdout_brier': brier,
    'lr_baseline': lr,
    # Arrays for evaluate.py
    'xgb_probs_holdout': xgb_probs_holdout,
    'calibrated_probs_holdout': cal_probs,
    'y_holdout': y_holdout,
    'train_df': train_df,
    'holdout_df': holdout_df,
    'X_train': X_train,
    'y_train': y_train,
    'X_holdout': X_holdout,
}

joblib.dump(bundle, 'model/wr_te_model_v2.pkl')
print('Saved model/wr_te_model_v2.pkl')
print(f'Final: AUC={auc:.4f}, Brier={brier:.4f}, tau={tau:.4f}')

## Step 7 — Sportsbook Comparison

In [ ]:
from scipy.special import logit as sp_logit

try:
    odds_df = pd.read_csv('data/anytime_td_odds.csv', index_col=0)
    odds_df.index.name = 'name'

    sportsbook_cols = [c for c in odds_df.columns if odds_df[c].notna().sum() > 0]
    odds_df['consensus_odds'] = odds_df[sportsbook_cols].mean(axis=1)
    odds_df = odds_df[odds_df['consensus_odds'].notna()].reset_index()
    print(f'Odds file: {len(odds_df)} players | Consensus from: {sportsbook_cols}')

    holdout_agg = (
        results_df
        .groupby('name')
        .agg(model_prob=('model_prob', 'mean'), scored_td=('scored_td', 'mean'))
        .reset_index()
    )

    merged = holdout_agg.merge(odds_df[['name', 'consensus_odds']], on='name', how='inner')
    merged['book_prob'] = merged['consensus_odds'].apply(american_to_implied_prob)
    merged['edge'] = merged['model_prob'] - merged['book_prob']
    merged['model_american'] = [prob_to_american(p) for p in merged['model_prob']]

    print(f'\nMatched {len(merged)} players to sportsbook odds')
    print('\nTop edges (model favors over book):')
    top = merged.sort_values('edge', ascending=False).head(20)
    print(top[['name', 'model_prob', 'book_prob', 'edge', 'model_american', 'consensus_odds']]
          .rename(columns={'consensus_odds': 'book_odds'})
          .to_string(index=False, float_format=lambda x: f'{x:.3f}'))

except FileNotFoundError:
    print('anytime_td_odds.csv not found')
except Exception as e:
    print(f'Error: {e}')